<a href="https://colab.research.google.com/github/edwardoughton/spatial_computing/blob/main/08_03_Loading_and_Processing_OSM_Data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🌍 🗺️ **Week 8: Working with OpenStreetMap data in GeoPandas — Part 3** 🌍 🗺️

In Parts 1 and 2 we processed administrative boundaries supplied with the course. Now we will download data from **OpenStreetMap (OSM)** directly into Jupyter, and apply the core `GeoPandas` processing toolkit you will need for your own research projects.

**Question:** How much of the City of Fairfax is within a short walk of a park, and how are everyday amenities distributed across the city?

We will cover:

1. Downloading a study area boundary and OSM features with `OSMnx`.
2. Inspecting and cleaning messy OSM data (mixed geometry types and many sparse columns).
3. Converting **polygons to points** (centroids and representative points).
4. Converting **points to polygons** (buffers, convex hulls, bounding boxes, and Voronoi polygons).
5. Clipping, **intersection**, **difference**, and dissolving.
6. **Spatial joins**, nearest-neighbor joins, and aggregating points to a grid.
7. Exporting several layers to one GeoPackage.

Run the examples in order. Task cells are for your own code. The final task asks you to repeat this workflow for **your own research topic**.

## 🧭 What is OpenStreetMap?

[OpenStreetMap](https://www.openstreetmap.org) is a free, editable map of the world built by volunteers. It is one of the most widely used open spatial datasets in research.

* OSM stores three **element** types: **nodes** (points), **ways** (lines or closed polygons), and **relations** (groups of elements, such as a park made of several parts).
* Every element is described by **tags**, which are `key=value` pairs. For example `amenity=cafe`, `leisure=park`, or `highway=residential`. Search for tags on the [OSM Map Features wiki](https://wiki.openstreetmap.org/wiki/Map_features) and see how commonly they are used on [taginfo](https://taginfo.openstreetmap.org/).
* Completeness varies by place and feature type. A missing cafe in OSM does not prove the cafe does not exist. Treat OSM as a *volunteered* dataset and state this as a limitation.
* OSM data is licensed under the [Open Database License](https://www.openstreetmap.org/copyright). Credit "© OpenStreetMap contributors" on maps and in reports.

We will use [`OSMnx`](https://osmnx.readthedocs.io/), which queries the OSM [Overpass API](https://wiki.openstreetmap.org/wiki/Overpass_API) and returns a `GeoDataFrame`. We will use it again later in the course for spatial networks and routing.

## 📦 Install and import packages

`OSMnx` is not pre-installed in Colab. Run the installation cell once per session.

In [ ]:
%pip install -q osmnx contextily

In [ ]:
from pathlib import Path
from urllib.request import urlretrieve

import contextily as ctx
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import osmnx as ox
from shapely.geometry import box

ox.settings.use_cache = True  # Save responses in ./cache so rerunning a cell does not download again.
ox.settings.requests_timeout = 60  # Give up on a busy server after 60 seconds (default 180).

# Set to False to skip the OSM servers and load the course snapshot of the same data.
use_live_osm = True
print('GeoPandas', gpd.__version__, '| OSMnx', ox.__version__)

## 🌐 📥 Define the study area

`ox.geocode_to_gdf()` sends a place query to the [Nominatim](https://nominatim.org/) geocoder and returns the matching OSM boundary as a `GeoDataFrame`.

**Be specific.** A plain text query such as `'City of Fairfax, Virginia, USA'` can match the *City of Fairfax Regional Library* instead of the city. A structured dictionary query with `city`, `state`, and `country` is more reliable. Always check what was returned before continuing: look at the name, the type, and the area.

In [ ]:
place = {'city': 'Fairfax', 'state': 'Virginia', 'country': 'USA'}
analysis_crs = 'EPSG:3968'  # NAD83 / Virginia Lambert, meter units (as in Parts 1 and 2)

# Fallback copy of the data used below (downloaded 28 October 2026), in case the OSM servers are busy.
snapshot_url = 'https://raw.githubusercontent.com/edwardoughton/spatial_computing/main/data/osm_fairfax.gpkg'
data_dir = Path('data')
data_dir.mkdir(exist_ok=True)
snapshot_path = data_dir / 'osm_fairfax.gpkg'


def load_snapshot(layer):
    """Read one layer of the course snapshot, downloading the file if needed."""
    if not snapshot_path.exists():
        urlretrieve(snapshot_url, snapshot_path)
    if layer not in gpd.list_layers(snapshot_path)['name'].values:
        raise ValueError(f'The course snapshot has no {layer!r} layer; retry the live download later.')
    return gpd.read_file(snapshot_path, layer=layer)


if use_live_osm:
    boundary = ox.geocode_to_gdf(place)[['osm_id', 'display_name', 'type', 'geometry']]
else:
    boundary = load_snapshot('boundary')

print(boundary[['display_name', 'type']].to_string(index=False))
print('CRS:', boundary.crs)
print('Area (km2):', round(boundary.to_crs(analysis_crs).area.iloc[0] / 1e6, 2))

The City of Fairfax is an independent city of roughly 16 km², next to George Mason University's campus. Plot it on a basemap to confirm we have the right place.

We use the Esri World Gray Canvas basemap, because its light background makes our data easy to see. Some popular tile services, including OpenStreetMap's own tiles and CARTO, now block scripted requests or require an API key.

In [ ]:
ax = boundary.plot(facecolor='none', edgecolor='crimson', linewidth=2, figsize=(8, 8))
ctx.add_basemap(ax, crs=boundary.crs, source=ctx.providers.Esri.WorldGrayCanvas)
ax.set_title('Study area: City of Fairfax, Virginia')
ax.set_axis_off()
plt.show()

## 📥 Download OSM features

`ox.features_from_polygon(polygon, tags)` downloads every OSM element inside the polygon that matches the `tags` dictionary:

* `{'building': True}` matches **any** value of the key (all buildings).
* `{'leisure': 'park'}` matches one value.
* `{'amenity': ['cafe', 'school']}` matches any value in the list.
* Several keys in one dictionary are combined with **OR**, not AND.

Overpass is a free, shared service. When many people send requests at the same time it may be slow or refuse queries. The function below tries a live download first and falls back to a course snapshot of the same layers. If downloads are very slow, set `use_live_osm = False` in the import cell and rerun, or try a different Overpass server, for example `ox.settings.overpass_url = 'https://maps.mail.ru/osm/tools/overpass/api'`. Read it carefully: it reuses ideas from the Functions week (arguments, return values, and error handling).

The raw output can have hundreds of columns, because every tag used by any feature becomes a column. We keep only the columns we need.

In [ ]:
study_polygon = boundary.geometry.iloc[0]


def load_osm_layer(layer, tags, columns):
    """Download OSM features inside the study area, keeping only selected columns."""
    if not use_live_osm:
        print(f'{layer}: loading the course snapshot')
        return load_snapshot(layer)
    try:
        raw = ox.features_from_polygon(study_polygon, tags)
        print(f'{layer}: {len(raw):,} features and {raw.shape[1]} columns downloaded from OSM')
        raw = raw.reset_index()  # Turn the (element, id) index into ordinary columns.
        keep = ['element', 'id'] + [col for col in columns if col in raw.columns] + ['geometry']
        return raw[keep]
    except Exception as error:
        print(f'{layer}: live download failed ({type(error).__name__}); using the course snapshot')
        return load_snapshot(layer)


amenity_types = ['school', 'restaurant', 'cafe', 'fast_food', 'library',
                 'place_of_worship', 'pharmacy', 'hospital', 'clinic']
amenities = load_osm_layer('amenities', {'amenity': amenity_types}, ['name', 'amenity'])
parks = load_osm_layer('parks', {'leisure': 'park'}, ['name', 'leisure'])
buildings = load_osm_layer('buildings', {'building': True}, ['name', 'building'])
bus_stops = load_osm_layer('bus_stops', {'highway': 'bus_stop'}, ['name', 'highway'])
road_types = ['motorway', 'trunk', 'primary', 'secondary', 'tertiary', 'residential']
roads = load_osm_layer('roads', {'highway': road_types}, ['name', 'highway'])

## 🔎 Inspect before processing

OSM layers are often **mixed**: a cafe may be mapped as a point (node), or as a building outline (way). Check geometry types and the `element` column before doing calculations.

In [ ]:
for name, layer in [('amenities', amenities), ('parks', parks), ('buildings', buildings),
                    ('bus_stops', bus_stops), ('roads', roads)]:
    print(f'{name:10s} CRS: {layer.crs} | geometry types: {layer.geom_type.value_counts().to_dict()}')

In [ ]:
print(amenities['amenity'].value_counts())
amenities.head()

## 🎯 🔎 Task: inspect the layers

* How many amenities have a missing `name`? (Hint: `.isna().sum()`.)
* Which amenity types are mapped as polygons, rather than points? (Hint: filter by `amenities.geom_type` and then use `value_counts()`.)
* Look at the roads layer. Why might a road query return a `Polygon`? Look up `area=yes` on the OSM wiki.

In [ ]:
# Enter your attempt here

## 🧭 Project and clean

All of our analysis uses distances and areas, so we project every layer to EPSG:3968 (meters). We then:

* keep only line geometries in `roads`, since some highway features are mapped as areas, and
* keep only polygon geometries in `parks` and `buildings`.

Filtering on `geom_type` like this is one of the most common OSM cleaning steps. Check what you drop: a few parks are mapped as a single point, which have no area to measure.

In [ ]:
boundary = boundary.to_crs(analysis_crs)
amenities = amenities.to_crs(analysis_crs)
bus_stops = bus_stops.to_crs(analysis_crs)

polygon_types = ['Polygon', 'MultiPolygon']
line_types = ['LineString', 'MultiLineString']
parks = parks[parks.geom_type.isin(polygon_types)].to_crs(analysis_crs)
buildings = buildings[buildings.geom_type.isin(polygon_types)].to_crs(analysis_crs)
roads = roads[roads.geom_type.isin(line_types)].to_crs(analysis_crs)

print('Parks:', len(parks), '| Buildings:', len(buildings), '| Road segments:', len(roads))

## ✂️ Clipping to the study area

The OSM query returns features that **touch** the polygon. Roads and parks that cross the city line extend outside it. `gpd.clip()` cuts geometries to the boundary, which is essential before measuring total length or area inside a study area.

In [ ]:
roads_clipped = gpd.clip(roads, boundary)
parks_clipped = gpd.clip(parks, boundary)

print('Road length before clipping (km):', round(roads.length.sum() / 1000, 1))
print('Road length after clipping (km): ', round(roads_clipped.length.sum() / 1000, 1))

ax = boundary.plot(facecolor='none', edgecolor='black', linewidth=2, figsize=(8, 8))
roads.plot(ax=ax, color='lightgray', linewidth=0.8)
roads_clipped.plot(ax=ax, color='steelblue', linewidth=0.8)
ax.set_title('Roads (gray) and roads clipped to the city (blue)')
ax.set_axis_off()
plt.show()

Lines have a `.length`; polygons have an `.area`. Use a `groupby()` to summarize road length by OSM road class.

In [ ]:
roads_clipped['length_km'] = roads_clipped.length / 1000
road_summary = roads_clipped.groupby('highway')['length_km'].sum().sort_values(ascending=False)
road_summary.round(1)

## 📍 Converting polygons to points

Many analyses need one point per feature, such as counting buildings in a grid cell or measuring distance between facilities.

* `.centroid` returns the geometric center. For a concave shape (such as an L-shaped building, or a park wrapped around a lake) the centroid can fall **outside** the polygon.
* `.representative_point()` returns a point that is guaranteed to be **inside** the polygon.

Use representative points when "inside" matters, such as joining a building to the park or grid cell it is in.

In [ ]:
park_centroids = parks_clipped.copy()
park_centroids['geometry'] = parks_clipped.centroid

park_points = parks_clipped.copy()
park_points['geometry'] = parks_clipped.representative_point()

outside = (~park_centroids.within(parks_clipped)).sum()
print(f'{outside} of {len(parks_clipped)} park centroids fall outside their own park polygon')

ax = parks_clipped.plot(color='lightgreen', edgecolor='green', figsize=(8, 8))
park_centroids.plot(ax=ax, color='red', markersize=15, label='Centroid')
park_points.plot(ax=ax, color='blue', markersize=15, marker='x', label='Representative point')
boundary.plot(ax=ax, facecolor='none', edgecolor='black')
ax.legend()
ax.set_title('Park centroids and representative points')
ax.set_axis_off()
plt.show()

### Standardizing a mixed layer to points

Our amenities include points and polygons. To count or map them consistently, convert every feature to a point. Points stay the same, and polygons become representative points.

In [ ]:
amenity_points = amenities.copy()
amenity_points['geometry'] = amenities.representative_point()
print('Before:', amenities.geom_type.value_counts().to_dict())
print('After: ', amenity_points.geom_type.value_counts().to_dict())

# A polygon that only touches the city can have its point outside it, so clip again.
amenity_points = gpd.clip(amenity_points, boundary)
print('Amenity points inside the city:', len(amenity_points), 'of', len(amenities))

## 🔵 Converting points to polygons

Points can become polygons in several ways, and each answers a different question:

| Method | Output | Typical use |
|---|---|---|
| `.buffer(distance)` | A circle around each point | Areas within a distance of a facility |
| `.union_all().convex_hull` | The smallest convex polygon enclosing all points | The extent of a set of observations |
| `.envelope` / `.total_bounds` | A rectangle | Bounding boxes and query windows |
| `.voronoi_polygons()` | One polygon per point, covering all the space nearest to it | Nearest-facility catchments |

We start with **buffers**. A common rule of thumb is that 400 m is roughly a five minute walk. Remember from Part 2 that this is straight line distance, not distance along streets. We will measure distance along the street network in the spatial networks week.

In [ ]:
walk_m = 400
bus_buffers = bus_stops.copy()
bus_buffers['geometry'] = bus_stops.buffer(walk_m)

ax = boundary.plot(facecolor='none', edgecolor='black', figsize=(8, 8))
bus_buffers.plot(ax=ax, alpha=0.2, color='orange')
bus_stops.plot(ax=ax, color='darkorange', markersize=5)
ctx.add_basemap(ax, crs=boundary.crs, source=ctx.providers.Esri.WorldGrayCanvas)
ax.set_title(f'{walk_m} m buffers around bus stops')
ax.set_axis_off()
plt.show()

The buffers overlap. As in Part 2, adding the buffer areas together counts shared space more than once. **Dissolve** them into one shape with `.union_all()` before measuring area.

In [ ]:
bus_area_summed = bus_buffers.area.sum() / 1e6
bus_coverage = gpd.GeoDataFrame(geometry=[bus_buffers.union_all()], crs=analysis_crs)
bus_area_unique = bus_coverage.area.iloc[0] / 1e6
print(f'Summed buffer area: {bus_area_summed:.1f} km2 (counts overlaps repeatedly)')
print(f'Dissolved area:     {bus_area_unique:.1f} km2')

### Convex hulls, bounding boxes, and Voronoi polygons

Below we use the schools in our amenity layer. The **convex hull** shows the extent of all schools; the **envelope** is the bounding rectangle; and **Voronoi polygons** divide space so that every location belongs to its nearest school. Clip the Voronoi polygons to the city so they do not extend to infinity.

Voronoi catchments are a geometric simplification. Real school attendance zones are set by policy, not distance.

In [ ]:
schools = amenity_points[amenity_points['amenity'] == 'school'].copy()
schools['name'] = schools['name'].fillna('Unnamed school')  # Rows with a missing value are not drawn by .plot(column=...).
print('Schools:', len(schools))

school_hull = gpd.GeoSeries([schools.union_all().convex_hull], crs=analysis_crs)
school_envelope = gpd.GeoSeries([schools.union_all().envelope], crs=analysis_crs)

voronoi = gpd.GeoDataFrame(geometry=schools.voronoi_polygons(extend_to=boundary.geometry.iloc[0]), crs=analysis_crs)
voronoi = gpd.clip(voronoi, boundary)
voronoi = voronoi.sjoin(schools[['name', 'geometry']], how='left', predicate='contains')  # Attach school names.

fig, axes = plt.subplots(1, 2, figsize=(14, 7))
boundary.plot(ax=axes[0], facecolor='none', edgecolor='black')
school_envelope.plot(ax=axes[0], facecolor='none', edgecolor='purple', linestyle='--')
school_hull.plot(ax=axes[0], alpha=0.3, color='purple')
schools.plot(ax=axes[0], color='black', markersize=15)
axes[0].set_title('Convex hull (shaded) and envelope (dashed)')

voronoi.plot(ax=axes[1], column='name', cmap='tab20', alpha=0.6, edgecolor='white')
boundary.plot(ax=axes[1], facecolor='none', edgecolor='black')
schools.plot(ax=axes[1], color='black', markersize=15)
axes[1].set_title('Voronoi polygons: nearest school catchments')
for ax in axes:
    ax.set_axis_off()
plt.show()

## 🔁 Converting polygons to lines (and lines to polygons)

* `.boundary` turns a polygon into its outline (a line). We used this in Part 1.
* `.buffer()` turns a line into a polygon corridor, for example the area within 50 m of a major road.

In [ ]:
major_roads = roads_clipped[roads_clipped['highway'].isin(['motorway', 'trunk', 'primary'])]
road_corridor = gpd.GeoDataFrame(geometry=[major_roads.buffer(50).union_all()], crs=analysis_crs)
city_outline = boundary.boundary

ax = road_corridor.plot(color='salmon', alpha=0.6, figsize=(8, 8))
major_roads.plot(ax=ax, color='darkred', linewidth=1)
city_outline.plot(ax=ax, color='black')
ax.set_title('50 m corridor around major roads, with the city outline')
ax.set_axis_off()
plt.show()

print('Share of city within 50 m of a major road (%):',
      round(100 * road_corridor.area.iloc[0] / boundary.area.iloc[0], 1))

## ✂️ Intersection and difference: who is near a park?

Now we answer our question. The steps are:

1. Buffer each park by 400 m and dissolve the buffers into one shape.
2. **Intersect** that shape with the city to find the part of the city within 400 m of a park.
3. Use **difference** to find the part of the city that is *not* near a park: the "park gaps".

We buffer the unclipped parks, so a park just outside the city line still counts for nearby residents. This is a deliberate choice; write down choices like this in your methods.

In [ ]:
park_walk_zone = gpd.GeoDataFrame(geometry=[parks.buffer(walk_m).union_all()], crs=analysis_crs)

near_park = boundary.overlay(park_walk_zone, how='intersection')
park_gaps = boundary.overlay(park_walk_zone, how='difference')

city_km2 = boundary.area.iloc[0] / 1e6
near_km2 = near_park.area.sum() / 1e6
gap_km2 = park_gaps.area.sum() / 1e6
print(f'City area:             {city_km2:.2f} km2')
print(f'Within {walk_m} m of a park: {near_km2:.2f} km2 ({100 * near_km2 / city_km2:.1f}%)')
print(f'Park gaps:             {gap_km2:.2f} km2 ({100 * gap_km2 / city_km2:.1f}%)')
print(f'Check (should be ~0):  {city_km2 - near_km2 - gap_km2:.6f}')

ax = near_park.plot(color='mediumseagreen', alpha=0.5, figsize=(8, 8))
park_gaps.plot(ax=ax, color='indianred', alpha=0.5)
parks_clipped.plot(ax=ax, color='darkgreen')
ctx.add_basemap(ax, crs=analysis_crs, source=ctx.providers.Esri.WorldGrayCanvas)
ax.set_title(f'Within {walk_m} m of a park (green) and park gaps (red)')
ax.set_axis_off()
plt.show()

## 🔗 Spatial joins

Area is only a rough proxy for people. Buildings are a better proxy for where people live and work. A **spatial join** (`.sjoin()`) attaches attributes from one layer to another based on their spatial relationship, without changing either geometry. See the [GeoPandas spatial joins guide](https://geopandas.org/en/stable/docs/user_guide/mergingdata.html#spatial-joins).

The `predicate` sets the relationship: `'intersects'` (default), `'within'`, `'contains'`, and others.

Below, we join each building's representative point to the park walking zone. Buildings with a match are near a park.

In [ ]:
building_points = buildings.copy()
building_points['geometry'] = buildings.representative_point()
building_points = gpd.clip(building_points, boundary)

joined = building_points.sjoin(park_walk_zone, how='left', predicate='within')
building_points['near_park'] = joined['index_right'].notna()

share = 100 * building_points['near_park'].mean()
print(f'{building_points["near_park"].sum():,} of {len(building_points):,} buildings ({share:.1f}%) are within {walk_m} m of a park')

**Think about it:** Is the share of buildings near a park higher or lower than the share of land area? Why might they differ? Which measure better answers our question?

### Nearest-neighbor joins

`.sjoin_nearest()` attaches the **closest** feature from another layer, and `distance_col` records the distance in CRS units (meters). Here we find the nearest bus stop to every school.

In [ ]:
school_bus = schools[['name', 'geometry']].sjoin_nearest(
    bus_stops[['geometry']], how='left', distance_col='bus_stop_m'
)
school_bus = school_bus.drop_duplicates(subset='name')  # Ties can create duplicate rows.
school_bus[['name', 'bus_stop_m']].sort_values('bus_stop_m').round(0)

### Aggregating points to a grid

To compare places fairly, researchers often count features in equal-sized cells. We create a 500 m grid, keep the cells that intersect the city, join the amenity points to the cells, and count them.

`.sjoin()` returns one row per match, so a `groupby()` on the grid cell ID gives a count. Cells with no amenities are missing from the join, so we fill them with zero.

In [ ]:
cell_m = 500
xmin, ymin, xmax, ymax = boundary.total_bounds
cells = [box(x, y, x + cell_m, y + cell_m)
         for x in np.arange(xmin, xmax, cell_m)
         for y in np.arange(ymin, ymax, cell_m)]
grid = gpd.GeoDataFrame({'cell_id': range(len(cells))}, geometry=cells, crs=analysis_crs)
grid = gpd.clip(grid, boundary)

amenity_cells = amenity_points.sjoin(grid, how='inner', predicate='within')
counts = amenity_cells.groupby('cell_id').size()
grid['amenities'] = grid['cell_id'].map(counts).fillna(0).astype(int)
print('Grid cells:', len(grid), '| Amenities counted:', grid['amenities'].sum(), 'of', len(amenity_points))

ax = grid.plot(column='amenities', cmap='YlOrRd', legend=True, edgecolor='white', linewidth=0.3,
               figsize=(9, 8), legend_kwds={'label': f'Amenities per {cell_m} m cell', 'shrink': 0.7})
boundary.plot(ax=ax, facecolor='none', edgecolor='black')
ax.set_title('Amenity counts on a 500 m grid')
ax.set_axis_off()
plt.show()

**Think about it:** Some cells at the edge are only partly inside the city. Why could this bias a comparison of counts between cells? How could you use `grid.area` to correct for it?

## 📤 Exporting results

A GeoPackage can store several layers in one file. Use the `layer=` argument when exporting and reading. Keep the OSM attribution and download date with your outputs, because OSM changes every day.

In [ ]:
output_dir = Path('outputs')
output_dir.mkdir(exist_ok=True)
output_path = output_dir / 'fairfax_osm_processed.gpkg'

boundary.to_file(output_path, layer='boundary', driver='GPKG')
near_park.to_file(output_path, layer='near_park', driver='GPKG')
park_gaps.to_file(output_path, layer='park_gaps', driver='GPKG')
grid.to_file(output_path, layer='amenity_grid', driver='GPKG')
building_points.to_file(output_path, layer='building_points', driver='GPKG')
road_summary.to_csv(output_dir / 'fairfax_road_length_km.csv')

print(gpd.list_layers(output_path))

## 🧰 Processing cheat sheet

| Task | GeoPandas code | Geometry in → out |
|---|---|---|
| Download OSM features | `ox.features_from_polygon(polygon, tags)` | → mixed |
| Keep one geometry type | `gdf[gdf.geom_type.isin(['Polygon', 'MultiPolygon'])]` | mixed → one type |
| Project | `gdf.to_crs('EPSG:3968')` | same |
| Clip to a boundary | `gpd.clip(gdf, boundary)` | same |
| Polygon → point | `gdf.centroid`, `gdf.representative_point()` | polygon → point |
| Point → polygon | `gdf.buffer(m)`, `.convex_hull`, `.envelope`, `.voronoi_polygons()` | point → polygon |
| Polygon → line | `gdf.boundary` | polygon → line |
| Line → polygon | `gdf.buffer(m)` | line → polygon |
| Dissolve | `gdf.union_all()` or `gdf.dissolve(by='column')` | many → one |
| Multipart → single parts | `gdf.explode(index_parts=False)` | multi → single |
| Intersection / difference | `a.overlay(b, how='intersection')`, `how='difference'` | new geometry |
| Spatial join | `a.sjoin(b, predicate='within')` | attributes only |
| Nearest join | `a.sjoin_nearest(b, distance_col='dist')` | attributes only |
| Length / area | `gdf.length`, `gdf.area` (projected CRS!) | → numbers |

## 🎯 🛒 Task: food access

Repeat the park analysis for supermarkets.

* Download `{'shop': 'supermarket'}` for the City of Fairfax using `load_osm_layer()`. (This layer is not in the snapshot, so it needs a live download.)
* Convert all supermarkets to points, buffer them by 800 m, and dissolve the buffers.
* Calculate the percentage of the city's area **and** the percentage of buildings within 800 m of a supermarket.
* Map the result and export it to a GeoPackage.
* Why might buffering the supermarkets *inside* the city alone underestimate access for residents near the city line? How would you fix it?

In [ ]:
# Enter your attempt here

## 🎯 🔬 Task: start your own research topic

Use this notebook as a template for your coursework project. Choose a place and a question that interests you. For example, cafes near universities, healthcare access in a rural county, tree cover in a neighborhood, or EV chargers along a corridor.

1. **Study area:** use `ox.geocode_to_gdf()` and check that the result is the place you meant (name, type, and area).
2. **Data:** choose two or more OSM layers. Look up the tags on the [Map Features wiki](https://wiki.openstreetmap.org/wiki/Map_features). Start with a small area, since large queries can take minutes.
3. **Clean:** project to a suitable CRS in meters (search [epsg.io](https://epsg.io/) for your region), filter geometry types, and clip to the study area.
4. **Process:** use at least **three** operations from the cheat sheet, such as polygons to points, buffer and dissolve, intersection or difference, a spatial join, or a grid count.
5. **Results:** produce a map, a summary table (CSV), and a GeoPackage.
6. **Interpret:** write three sentences: a finding, a methodological choice you made, and a limitation of OSM data for your question.

In [ ]:
# Enter your attempt here

## ✅ Before you finish

* OSM is volunteered data. Its completeness varies by place and feature type. Cite "© OpenStreetMap contributors" and record your download date.
* Always check the geocoded study area, the geometry types, and the CRS before calculating.
* Dissolve overlapping buffers before measuring area, so that shared space is only counted once.
* Straight line buffers ignore streets, rivers, and highways that block walking. In the spatial networks week we will download the street network with `OSMnx` and measure distance along it.

See the [OSMnx features documentation](https://osmnx.readthedocs.io/en/stable/user-reference.html#module-osmnx.features), the [GeoPandas set operations guide](https://geopandas.org/en/stable/docs/user_guide/set_operations.html), and the [GeoPandas geometric manipulations guide](https://geopandas.org/en/stable/docs/user_guide/geometric_manipulations.html).